# Week 2 Outliers and Data Quality - Solution

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/obscrivn/mynewbook/blob/master/module2/week2_outlier_practice_solution.ipynb)


**Solution · Estimated time: 20–30 minutes**

You are reviewing one month of delivery records for a regional courier. A few delivery times look unusual. Your job is to use the **interquartile range (IQR)** method to flag potential outliers, inspect their context, and recommend a defensible data-quality action.

> **Core principle:** An outlier flag is a question, not a deletion command. An extreme value may be an error, a rare but legitimate event, or an important case.

## Learning objectives

By the end of this practice, you will be able to:

1. Use a boxplot to identify possible extreme values.
2. Calculate Q1, Q3, the IQR, and the 1.5 × IQR bounds with pandas.
3. Filter and inspect rows flagged as potential outliers.
4. Use domain context to distinguish unusual observations from likely data-quality problems.
5. Explain when you might **retain, correct, transform, or remove** an observation.

### Suggested pacing

- Set up and inspect the data: 5 minutes
- Visualize and calculate IQR bounds: 8 minutes
- Investigate flagged records: 8 minutes
- Decide and reflect: 5 minutes

## 1. Set up and load the data

The dataset is stored in this course's GitHub repository. The code below uses the file's **raw URL**, which lets pandas read the CSV directly in Google Colab or Jupyter. An internet connection is required. A local copy is also provided as `week2_outlier_data.csv`.

We will use only **pandas** for tabular analysis and **matplotlib** for plots.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

data_url = (
    "https://raw.githubusercontent.com/obscrivn/DataScience-book/"
    "main/module02/week2_outlier_data.csv"
)

deliveries = pd.read_csv(data_url)
deliveries.head()

### Data dictionary

| Column | Meaning |
|---|---|
| `delivery_id` | Unique record identifier |
| `route_type` | Urban, suburban, or rural route |
| `distance_km` | Route distance in kilometers |
| `weather` | Recorded weather condition |
| `packages` | Number of packages delivered |
| `delivery_minutes` | Total delivery duration; this is our target variable |
| `source_note` | Information about how the record was captured or checked |

In [ ]:
print(f"Rows: {deliveries.shape[0]}")
print(f"Columns: {deliveries.shape[1]}")

deliveries.info()

### Think

Before calculating anything, which columns could help you decide whether an unusually long or short delivery is believable? Why?

**Possible response:** `distance_km`, `route_type`, and `weather` provide operational context. `source_note` also helps distinguish a verified event from a possible recording problem.

## 2. Describe and visualize delivery time

Start with descriptive statistics. The median is the 50th percentile. Q1 and Q3 are the 25th and 75th percentiles.

In [ ]:
deliveries["delivery_minutes"].describe()

### Predict

Before running the next cell, where do you expect most of the boxplot to appear? How many individual points might appear beyond the whiskers?

**Possible response:** Most values should cluster below about 80 minutes. I would expect roughly three points beyond the whiskers: one very short time and two very long times.

In [ ]:
fig, ax = plt.subplots(figsize=(9, 3))
ax.boxplot(deliveries["delivery_minutes"], vert=False)
ax.set_title("Delivery time distribution")
ax.set_xlabel("Delivery time (minutes)")
ax.set_yticks([])
plt.show()

### Reading a boxplot

- The line inside the box is the **median**.
- The left and right edges of this horizontal box are **Q1** and **Q3**.
- The box length is the **interquartile range: IQR = Q3 − Q1**.
- The whiskers extend to the most extreme observed values still within the chosen rule.
- Points beyond the whiskers are **potential outliers**, not automatically mistakes.

A boxplot summarizes one variable. It cannot tell us whether a value makes sense for a particular route.

## 3. Calculate the IQR bounds

The familiar 1.5 × IQR rule is:

\[
	ext{lower bound} = Q1 - 1.5(IQR)
\]

\[
	ext{upper bound} = Q3 + 1.5(IQR)
\]

Values strictly below the lower bound or strictly above the upper bound will be flagged.

In [ ]:
delivery_times = deliveries["delivery_minutes"]

q1 = delivery_times.quantile(0.25)
q3 = delivery_times.quantile(0.75)
iqr = q3 - q1

lower_bound = q1 - 1.5 * iqr
upper_bound = q3 + 1.5 * iqr

iqr_summary = pd.Series({
    "Q1": q1,
    "Q3": q3,
    "IQR": iqr,
    "lower_bound": lower_bound,
    "upper_bound": upper_bound,
})
iqr_summary

### Interpret

Explain Q1, Q3, and the upper bound in the context of delivery time. Is the upper bound the same as a maximum allowed delivery time?

**Possible response:** Q1 = 33 minutes and Q3 = 52 minutes, so the middle 50% spans 19 minutes. The upper IQR bound is 80.5 minutes. It is a statistical flagging threshold for this sample—not a service target, physical limit, or policy cutoff.

## 4. Flag and inspect potential outliers

The Boolean mask below keeps the IQR rule separate from the data. This makes the logic easy to inspect and reuse.

In [ ]:
is_potential_outlier = (
    (delivery_times < lower_bound)
    | (delivery_times > upper_bound)
)

potential_outliers = deliveries.loc[is_potential_outlier].copy()
potential_outliers = potential_outliers.sort_values("delivery_minutes")

potential_outliers

In [ ]:
print(f"Flagged rows: {is_potential_outlier.sum()} of {len(deliveries)}")

potential_outliers[
    [
        "delivery_id",
        "delivery_minutes",
        "distance_km",
        "route_type",
        "weather",
        "source_note",
    ]
]

### Think

Which flagged row looks most plausibly legitimate? Which looks most like a data-quality problem? What evidence supports each judgment?

**Possible response:** W2-031 (165 minutes) is plausibly legitimate because it is a verified 68.4 km rural route in snow. W2-033 (4 minutes for 14.8 km) looks suspicious because the implied trip is implausibly fast. W2-032 (240 minutes for 5.6 km) also needs investigation, but a severe delay is still possible.

### Add context before deciding

Compare typical values within each route type. Group summaries do not prove whether a flagged row is correct, but they can make your questions more precise.

In [ ]:
route_summary = deliveries.groupby("route_type")["delivery_minutes"].agg(
    ["count", "median", "min", "max"]
)
route_summary

### Interpret

Does comparing route types change your view of the 165-minute record? What source would you check next for the 4- and 240-minute records?

**Possible response:** The rural group normally takes longer, which makes 165 minutes less surprising even though it remains extreme. For the other records, I would check the original scanner/GPS event log, timestamps, unit conventions, or an operations report before editing anything.

## 5. Record a decision without changing the raw data

Good data cleaning is traceable. Keep the original rows and create a small review table. Use one of these labels in the `decision` column:

- `retain`
- `correct after verification`
- `transform for analysis`
- `remove from a specific analysis`
- `investigate`

For now, the student version intentionally leaves the recommendations blank. Fill them in using evidence from the row—not its extremeness alone.

In [ ]:
decisions = potential_outliers[
    ["delivery_id", "delivery_minutes", "distance_km", "weather", "source_note"]
].copy()

decision_map = {
    "W2-031": "retain",
    "W2-032": "investigate",
    "W2-033": "correct after verification",
}

rationale_map = {
    "W2-031": "Verified long rural route in snow; extreme but plausible.",
    "W2-032": "A four-hour urban trip is possible, but the manual entry needs checking.",
    "W2-033": "Four minutes for 14.8 km is implausible; confirm a likely entry/import error.",
}

decisions["decision"] = decisions["delivery_id"].map(decision_map)
decisions["rationale"] = decisions["delivery_id"].map(rationale_map)
decisions

### Filtering is a comparison, not a final cleaning decision

The next cell creates an analysis copy that excludes flagged rows. The original `deliveries` DataFrame is unchanged. Comparing both views can reveal how influential the flagged observations are.

In [ ]:
unflagged_deliveries = deliveries.loc[~is_potential_outlier].copy()

comparison = pd.DataFrame({
    "all_rows": deliveries["delivery_minutes"].describe(),
    "unflagged_only": unflagged_deliveries["delivery_minutes"].describe(),
})
comparison

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(10, 3), sharey=True)

axes[0].boxplot(deliveries["delivery_minutes"])
axes[0].set_title("All rows")
axes[0].set_ylabel("Delivery time (minutes)")
axes[0].set_xticks([])

axes[1].boxplot(unflagged_deliveries["delivery_minutes"])
axes[1].set_title("Flagged rows excluded")
axes[1].set_xticks([])

plt.tight_layout()
plt.show()

### Predict / Interpret

How did filtering change the mean, standard deviation, and maximum? Would the filtered data answer the same business question as the full data?

**Possible response:** Filtering sharply lowers the mean, standard deviation, and maximum because the extreme times are influential. The datasets may answer different questions: excluding exceptions can describe routine operations, while retaining them is essential for studying delays, customer experience, or operational risk.

## 6. Final reflection: retain, correct, transform, or remove?

Possible responses:

1. **Retain:** Keep a verified extreme such as W2-031 because long rural routes in snow are part of real operations and may matter most when studying delay risk.
2. **Correct:** Replace a value only when a trustworthy source confirms the intended value—for example, if the raw timestamp proves that `4` should have been `40`.
3. **Transform:** Use a log scale when a valid right-skewed duration variable overwhelms a visualization or violates an analysis assumption; the original value should remain available.
4. **Remove:** Exclude a row from a named analysis when it is proven invalid or outside that analysis's target population, and document the rule and impact.

The IQR rule only identifies values that are unusual relative to this sample. It uses no knowledge of distance, weather, measurement processes, or the business question, so a flag alone is not evidence of error.

## Takeaways

- A boxplot and the 1.5 × IQR rule are useful **screening tools**.
- Q1 and Q3 describe the middle half of the observed distribution; the IQR measures its spread.
- Always inspect complete flagged rows and relevant group context.
- Preserve raw data and document any correction, transformation, or exclusion.
- The right action depends on evidence and the question you are trying to answer.